# 03 · Mesh and receivers

This notebook builds the inversion mesh and the receiver file that goes with it:

1. choose the cell sizes of a coarse starting grid,
2. refine it where the data need small cells (around the survey, along the coast, at the surface,
   around each station, and anywhere else you like),
3. fit it to the topography (and bathymetry),
4. add the ocean,
5. put the stations on the surface and write the files.

Each refinement is a plain call to pyGoFEM in its own cell, so you can change it, repeat it,
remove it or add your own. The steps around the refinements call the same functions as the setup
GUI.

* **Needs:** EDI files, and from notebook 02 `model_frame.json` and `DEM_projected_gridded.npz`.
* **Writes** into `work/03_mesh/`: `model.tria` (the mesh for GoFEM), `model.vtu` (to look at in
  ParaView), `model.materials`, `model.mask` and `receivers_projected.csv`.
* **Takes:** about a minute for the sample (most of it in the topography fit); real surveys with
  more stations and smaller cells take longer.
* **GUI:** setup tab 5, *Mesh*.

No data file is written: select the data in `gateway_mt`, using the receiver file from here.

## How refinement works

deal.II meshes are refined by splitting a cell into 8 (halving it in every direction). Each
refinement call below splits every cell that touches a region once. Cells next to each other may
differ by one level at most, so neighbours get split too where needed, which makes a smooth
transition from small to large cells. More levels in a region means smaller cells there: each
level halves the size. The cell count roughly grows 8 times for each level over the same volume,
so the inner, finest regions should be the smallest.

## Settings

In [ ]:
EDI_DIR = "sample_data/edis"                       # the stations (the same ones as in notebook 02)
EXCLUDE = []                                       # station names to leave out
FRAME_FILE = "work/02_survey/model_frame.json"     # from notebook 02
DEM_FILE = "work/02_survey/DEM_projected_gridded.npz"   # from notebook 02; None for a flat surface
OUT_DIR = "work/03_mesh"

# --- resistivities (ohm-m) ----------------------------------------------------------------
BACKGROUND_RHO = 100.0   # the starting model's earth resistivity
# Typical apparent resistivity at the shortest and longest periods. Only used to size the cells
# (through the skin depth); read them off your data in gateway_mt.
RHO_SHALLOW = 100.0
RHO_DEEP = 100.0

# --- surface and ocean --------------------------------------------------------------------
USE_TOPOGRAPHY = True    # fit the land surface to the DEM (False: flat at sea level)
# "none": no ocean
# "thin_sheet": the ocean as a thin conductive layer at the surface, with the conductance of the
#               real water column (simple and robust; the sea floor shape isn't in the mesh)
# "real": water cells above the real sea floor (needs small cells to resolve the bathymetry)
OCEAN = "thin_sheet"
OCEAN_RHO = 0.3          # sea water (ohm-m)
SHEET_THICKNESS_M = 1500.0   # thin sheet only: the layer that carries the ocean's conductance
# Which DEM points below sea level are sea:
# "connected": joined to the open sea (the DEM's edge). Land below sea level elsewhere (Death
#              Valley, polders, lake beds) stays land, at its own elevation, with air above it.
# "all":       everything below sea level.
SEA_RULE = "connected"
LAND_POLYGONS = None     # optional vector file: polygons that are never sea
SEA_POLYGONS = None      # optional vector file: polygons that are sea wherever below sea level

# --- output ---------------------------------------------------------------------------------
# The GoFEM files are always written. True also writes the same mesh for FEMTIC (mesh.dat +
# resistivity_block_iter0.dat, its DHEXA format). FEMTIC takes OCEAN = "none" or "real" only.
FEMTIC = False
# True also writes a SimPEG octree (discretize TreeMesh, UBC files + simpeg_model.npz): as fine as
# the GoFEM cells horizontally, with earth / air / sea cells from the DEM. "none" or "real" only.
SIMPEG = False
SIMPEG_VERTICAL_RATIO = 1.0   # octree cells are this many times wider than tall (1 = cubes)

# --- optional fixed mantle -----------------------------------------------------------------
MANTLE_DEPTH_KM = None   # e.g. 410: cells below this get MANTLE_RHO and are not inverted
MANTLE_RHO = 20.0

In [ ]:
%matplotlib inline
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mtinv_gui.backends import edi_io
from mtinv_gui.backends import mesh as M
from mtinv_gui.backends import mesh_design as D
from mtinv_gui.backends import mesh_dem
from mtinv_gui.backends import mesh_plots
from mtinv_gui.backends import projection as P

os.makedirs(OUT_DIR, exist_ok=True)

## 1. Stations, model frame and DEM

The stations go into the model frame as **(north, east, z)** in metres: the mesh's x axis points
north (GoFEM's convention). `receivers` keeps them in that order for the rest of the notebook.

`dem` is the projected DEM as the build uses it: the detailed grid, the regional grid (if
notebook 02 made one), and where the sea is. `dem.elevation(points)` gives the elevation at
mesh-frame points `[[north, east], ...]`; `dem.inland(points)` says where land is below sea level.

In [ ]:
frame = P.load_frame(FRAME_FILE)
print("model frame:", P.frame_label(frame))

stations = [st for st in edi_io.read_edi_headers(EDI_DIR) if st.station_id not in set(EXCLUDE)]
receivers, names = P.project_stations(stations, frame, swap_xy=True)    # [[north, east, 0], ...]
rxy = np.asarray(receivers)[:, :2]
spacing = P.station_spacing_stats(receivers)
period_min = min(st.period_min_s for st in stations)
period_max = max(st.period_max_s for st in stations)
print(f"{len(names)} stations, spacing {spacing['min_m'] / 1e3:.1f}-{spacing['max_m'] / 1e3:.1f} km, "
      f"periods {period_min:g}-{period_max:g} s")

if DEM_FILE:
    dem = M.read_mesh_dem(DEM_FILE, sea_rule=SEA_RULE, land_polygons_path=LAND_POLYGONS or "",
                          sea_polygons_path=SEA_POLYGONS or "", frame=frame)
    tc = dem.topo_cart                          # the detailed grid: [..., 0] east, [..., 1] north, [..., 2] elevation
    footprint = (np.abs(tc[..., 0]) <= np.abs(rxy[:, 1]).max()) & (np.abs(tc[..., 1]) <= np.abs(rxy[:, 0]).max())
    relief = float(np.ptp(tc[..., 2][footprint]))
    print(f"relief under the survey {relief:.0f} m")
    print(f"land below sea level (kept as land): {dem.ocean_stats()['inland_km2']:.0f} km²")
else:
    dem = None
    relief = 0.0
    if OCEAN != "none" or USE_TOPOGRAPHY:
        print("no DEM: building with a flat surface and no ocean")
        OCEAN, USE_TOPOGRAPHY = "none", False

## 2. Cell sizes of the starting grid

The suggestion follows the same rules as the GUI:
* the model reaches 3 skin depths of the longest period below and beyond the stations;
* the base cells are coarse, and the refinements below make the small cells near the survey.

The skin depth is about 0.5 &times; sqrt(resistivity &times; period) km.

In [ ]:
rules = D.DesignRules(mantle_transition=MANTLE_DEPTH_KM is not None,
                      mantle_depth_km=MANTLE_DEPTH_KM or 410.0, mantle_resistivity_ohm_m=MANTLE_RHO)
suggestion = D.suggest_mesh(
    footprint_x_m=float(np.ptp(rxy[:, 0])), footprint_y_m=float(np.ptp(rxy[:, 1])),
    spacing_min_m=spacing["min_m"], spacing_median_m=spacing["median_m"],
    period_min_s=period_min, period_max_s=period_max,
    rho_shallow=RHO_SHALLOW, rho_deep=RHO_DEEP, relief_m=relief, rules=rules)
meta = suggestion.meta
print(f"skin depth: {meta['delta_min_m'] / 1e3:.2f} km at {period_min:g} s, "
      f"{meta['delta_max_m'] / 1e3:.0f} km at {period_max:g} s")
print("suggested horizontal cells (centre outwards, m):", suggestion.dx_blocks)
print("suggested earth layers (surface down, m):", suggestion.dz_earth)
pd.DataFrame([{"check": d.name, "value": round(d.value), "target": d.target, "ok": d.ok}
              for d in suggestion.diagnostics])

To use your own sizes, replace a suggestion below with a list of numbers in metres, e.g.
`DZ_EARTH = [5000, 5000, 10000, 20000, 40000, 80000, 160000]`.

In [ ]:
DX_BLOCKS = suggestion.dx_blocks   # horizontal cells from the centre outwards (mirrored, same north and east)
DZ_EARTH = suggestion.dz_earth     # earth layers from the surface down
DZ_AIR = suggestion.dz_air         # air layers from the surface up (keep the same total as the earth)

dx, dy, dz = M.cell_size_lists(DX_BLOCKS, DZ_AIR, DZ_EARTH)
bounds = M.model_bounds(dx, dy)    # (east min, east max, north min, north max)
summary = M.initial_grid_summary(DX_BLOCKS, DZ_AIR, DZ_EARTH)
print(f"model: {summary['extent_x_m'] / 1e3:.0f} km square, {summary['earth_thickness_m'] / 1e3:.0f} km deep, "
      f"{summary['n_cells_initial']:,} cells before refinement")

if dem is not None:
    short = M.warn_dem_shortfall(dem, bounds, OCEAN)    # prints a warning if it doesn't reach
    if not any(short.values()):
        print("the DEM covers the whole mesh")

## 3. The starting grid and its materials

`prepare_pygofem()` makes deal.II and pyGoFEM importable. The grid is centred on (0, 0, 0) with z
pointing down. Cells above z = 0 are air (material 1), below it earth (material 2).

In [ ]:
M.prepare_pygofem()
from gofem.mesh_utils import refine_around_points, refine_at_interface, refine_within_polygon

tria = M.base_grid(dx, dy, dz)
materials = M.initial_materials(
    tria, BACKGROUND_RHO,
    mantle_depth_m=MANTLE_DEPTH_KM * 1e3 if MANTLE_DEPTH_KM else None,
    mantle_resistivity_ohm_m=MANTLE_RHO)
print("materials:", materials)


def report(label):
    """Print the cell count and size range after a step."""
    print(f"{label}: {tria.n_active_cells():,} cells, "
          f"smallest {tria.minimal_cell_diameter():,.0f} m, largest {tria.maximal_cell_diameter():,.0f} m")


report("starting grid")

## 4. Refinements

### 4a. Around the survey

Nested regions around the stations ("shells"), each refined once, from the largest to the
smallest. Each one is an ellipse around the stations' bounding box, grown by a buffer, and goes
down to a depth. Cells keep halving inwards, so the innermost shell has the smallest cells.

`SURVEY_SHELLS_KM` is a list of `(buffer, depth)` pairs in km, outer to inner. `None` uses the
suggestion printed below.

In [ ]:
SURVEY_SHELLS_KM = None    # e.g. [(180, 180), (90, 100), (45, 55)]

if SURVEY_SHELLS_KM is None:
    buffers, depths = D.survey_area_shells(suggestion.stages[0], model_depth_m=sum(DZ_EARTH))
    SURVEY_SHELLS_KM = [(round(b, 1), round(d, 1)) for b, d in zip(buffers, depths)]
print("survey shells (buffer km, depth km):", SURVEY_SHELLS_KM)

centre_n = 0.5 * (rxy[:, 0].min() + rxy[:, 0].max())
centre_e = 0.5 * (rxy[:, 1].min() + rxy[:, 1].max())
half_n = 0.5 * np.ptp(rxy[:, 0])
half_e = 0.5 * np.ptp(rxy[:, 1])

for buffer_km, depth_km in SURVEY_SHELLS_KM:
    refine_around_points(
        tria,
        points=[[centre_n, centre_e, 0.0]],          # one point: the survey centre
        center=[centre_n, centre_e, 0.0],
        radii=[half_n + buffer_km * 1e3, half_e + buffer_km * 1e3, depth_km * 1e3],   # north, east, depth (m)
        repeat=1,
        exclude_materials=[M.AIR_ID],                # don't refine the air
        pnorm=2)                                     # 2 = ellipse; 1 = rectangle
    report(f"  shell +{buffer_km} km, to {depth_km} km")

**Other shapes.** For a box, use `pnorm=1` above. To follow the outline of the stations instead,
refine within a polygon (here the convex hull of the stations, grown by each buffer):
```python
import shapely.geometry as sg
hull = sg.MultiPoint([tuple(p) for p in rxy]).convex_hull          # in (north, east) metres
for buffer_km, depth_km in SURVEY_SHELLS_KM:
    refine_within_polygon(tria, polygon=hull.buffer(buffer_km * 1e3), repeat=1,
                          z_range=[0.0, depth_km * 1e3], pnorm=1)   # pnorm 2: a rounded bottom
```
Any shapely polygon in (north, east) metres works, e.g. one read from a shapefile and projected
with `projection.project_lonlat(lons, lats, frame, swap_xy=True)`.

### 4b. Along the coastline (with an ocean)

The coastline is the edge of the sea (land below sea level doesn't count). Small islands and
inlets are dropped and the line is simplified, then cells within `COAST_RADIUS_KM` of it (down to
`COAST_DEPTH_KM`) are refined.
This resolves the strong conductivity contrast at the coast. Each further level halves the depth
again.

In [ ]:
COAST_LEVELS = 1           # 0 to skip
COAST_RADIUS_KM = 10.0     # how far from the coast to refine
COAST_DEPTH_KM = 10.0      # how deep (first level)
COAST_SIMPLIFY_KM = 2.0    # smooth the coastline to this tolerance
COAST_MIN_AREA_KM2 = 50.0  # drop islands and lakes smaller than this

coast_points = []
if OCEAN != "none" and COAST_LEVELS > 0:
    stage = {"radius_km": COAST_RADIUS_KM, "simplify_km": COAST_SIMPLIFY_KM,
             "min_area_km2": COAST_MIN_AREA_KM2}
    grow = COAST_RADIUS_KM * 1e3
    coast_points, info = dem.coast_points(
        stage, (bounds[0] - grow, bounds[1] + grow, bounds[2] - grow, bounds[3] + grow))
    print(f"coastline: {info['n_lines']} lines, {info['length_m'] / 1e3:.0f} km, "
          f"{len(coast_points)} points every {info['spacing_m'] / 1e3:.1f} km")
    for level in range(COAST_LEVELS):
        M.refine_coast(
            tria, coast_points,
            radii=[COAST_RADIUS_KM * 1e3, COAST_RADIUS_KM * 1e3, COAST_DEPTH_KM * 1e3 * 0.6 ** level],
            pnorm=2,
            # the real ocean's water is still "air" here, so it must be refined too
            exclude=() if OCEAN == "real" else (M.AIR_ID,))
        report(f"  coast level {level + 1}")

    cp = np.asarray(coast_points)
    fig, ax = plt.subplots(figsize=(7, 6))
    for g, inland in zip(dem.grids[::-1], dem.inland_masks[::-1]):    # the regional grid first, under
        ax.pcolormesh(g[..., 0] / 1e3, g[..., 1] / 1e3, g[..., 2],
                      cmap="terrain", vmin=-4000, vmax=3000, shading="auto")
        if inland.any():
            ax.contour(g[..., 0] / 1e3, g[..., 1] / 1e3, inland.astype(float), [0.5],
                       colors="purple", linewidths=1.0)
    ax.plot([], [], color="purple", label="land below sea level")
    ax.plot(cp[:, 1] / 1e3, cp[:, 0] / 1e3, ".", color="m", ms=3, label="coast refinement points")
    ax.plot(rxy[:, 1] / 1e3, rxy[:, 0] / 1e3, "kv", ms=5, mfc="w", label="stations")
    ax.set_xlim(bounds[0] / 1e3, bounds[1] / 1e3)
    ax.set_ylim(bounds[2] / 1e3, bounds[3] / 1e3)
    ax.set_aspect("equal")
    ax.set_xlabel("east (km)")
    ax.set_ylabel("north (km)")
    ax.legend(loc="lower left")
    ax.set_title("the coastline used for refinement, whole model")
    plt.show()

### 4c. At the surface

Refines the earth cells at the surface (the boundary between earth and air) everywhere, which
gives thin cells where the topography is fitted and where the shortest periods are sensitive.
With the real ocean, the water cells above the sea floor are refined too.

In [ ]:
SURFACE_LEVELS = 2

for level in range(SURFACE_LEVELS):
    refine_at_interface(tria, material_ids=[M.EARTH_ID], repeat=1,
                        center=[0, 0, 0], radii=[1e7, 1e7], pnorm=1)
    if OCEAN == "real":
        M.refine_air_over_ocean(tria, dem.ocean_elevation, M.AIR_ID, [M.EARTH_ID])
    report(f"  surface level {level + 1}")

### 4d. Around each station

Small ellipsoids around every station, for the cells right under them.

In [ ]:
STATION_LEVELS = 1         # 0 to skip
STATION_RADIUS_KM = 5.0
STATION_DEPTH_KM = 10.0

for level in range(STATION_LEVELS):
    refine_around_points(tria, points=receivers, center=[0, 0, 0],
                         radii=[STATION_RADIUS_KM * 1e3, STATION_RADIUS_KM * 1e3, STATION_DEPTH_KM * 1e3],
                         repeat=1, exclude_materials=[M.AIR_ID], pnorm=2)
    report(f"  stations level {level + 1}")

### 4e. Your own refinement

Anything that can decide, cell by cell, whether to split it: set `cell.refine_flag = "isotropic"`
on the cells to split, then call `tria.execute_coarsening_and_refinement()`. Cell centres are
`cell.center().x` (north), `.y` (east) and `.z` (depth, before the topography fit), in metres.

The example refines a 10 km wide band along a fault trace, down to 20 km. Set `DO_CUSTOM = False`
to skip it.

In [ ]:
DO_CUSTOM = True
FAULT_KM = [(40.0, -50.0), (-40.0, 10.0)]    # fault trace end points, (north, east) in km
FAULT_HALF_WIDTH_KM = 5.0
FAULT_DEPTH_KM = 20.0

if DO_CUSTOM:
    a, b = np.array(FAULT_KM[0]) * 1e3, np.array(FAULT_KM[1]) * 1e3
    n_flagged = 0
    for cell in tria.active_cells():
        c = cell.center()
        if cell.material_id == M.AIR_ID or c.z > FAULT_DEPTH_KM * 1e3:
            continue
        p = np.array([c.x, c.y])
        t = np.clip(np.dot(p - a, b - a) / np.dot(b - a, b - a), 0.0, 1.0)   # nearest point on the trace
        if np.linalg.norm(p - (a + t * (b - a))) < FAULT_HALF_WIDTH_KM * 1e3:
            cell.refine_flag = "isotropic"
            n_flagged += 1
    tria.execute_coarsening_and_refinement()
    report(f"  fault band ({n_flagged:,} cells flagged)")

### The refined mesh

Coloured by cell width: a plan view just below sea level and an east-west section through the
middle of the survey. Look for a smooth growth of the cells away from the stations, and small
enough cells under each one (a few across the station spacing).

In [ ]:
cells = mesh_plots.mesh_cells(tria)
fig, (ax_plan, ax_sec) = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={"width_ratios": [1, 1.5]})
mesh_plots.plot_plan(cells, 500.0, ax=ax_plan, stations=receivers, color_by="size")
mesh_plots.plot_section(cells, north_m=centre_n, ax=ax_sec, stations=receivers, color_by="size",
                        range_km=((centre_e - half_e) / 1e3 - 150, (centre_e + half_e) / 1e3 + 150),
                        depth_km=(-3, 100))
plt.show()

## 5. Fit to the topography

Moves the mesh's nodes up or down so the earth's surface follows the DEM: land (including land
below sea level), and the sea floor with the real ocean. This takes a while. A warning about
degenerate cells means some cells were squashed flat: refine the surface more (4c), or use a
smoother DEM.

In [ ]:
if dem is not None and (USE_TOPOGRAPHY or OCEAN == "real"):
    fit = M.fit_topography(tria, dem, land=USE_TOPOGRAPHY, bathymetry=(OCEAN == "real"))
else:
    print("no topography fit: the surface stays flat at sea level")

## 6. The ocean

* **thin sheet:** the cells in the top `SHEET_THICKNESS_M` over the sea get a conductivity that
  gives them the conductance of the water column above, grouped into about 100 materials.
* **real:** air cells that ended up below sea level after the fit become sea water, except over
  land below sea level. The notebook
  compares the water volume in the mesh with the DEM's: well below 100 % means the sea floor isn't
  resolved, so add surface or coast refinement.

Ocean cells are fixed: the inversion doesn't change them.

In [ ]:
ocean_stats = {}
if OCEAN != "none":
    ocean_stats = M.add_ocean(tria, OCEAN, dem, bounds=bounds, dz_earth=DZ_EARTH,
                              materials=materials, background_resistivity_ohm_m=BACKGROUND_RHO,
                              ocean_resistivity_ohm_m=OCEAN_RHO, sheet_m=SHEET_THICKNESS_M)
print(f"{len(materials)} materials")

## 7. Put the stations on the surface

Each station is moved down (or up) onto the mesh's surface; seafloor stations with the real ocean
go onto the sea floor. The table compares the station's height in the mesh with its EDI header
and the DEM. The mesh surface is flat across each cell, so on steep ground with large surface
cells it can differ from the DEM by a few hundred metres. Refine more around the stations (4d) if
that matters for your data.

In [ ]:
seafloor = bool(ocean_stats.get("mode") == "real" and ocean_stats.get("n_ocean_cells"))
receivers_projected = np.asarray(M.project_receivers(tria, receivers, seafloor=seafloor))

check = pd.DataFrame({"station": names,
                      "north_km": (receivers_projected[:, 0] / 1e3).round(2),
                      "east_km": (receivers_projected[:, 1] / 1e3).round(2),
                      "mesh_elev_m": (-receivers_projected[:, 2]).round(1),
                      "edi_elev_m": [st.elev for st in stations]})
if dem is not None:
    check["dem_elev_m"] = np.round(dem.elevation(receivers_projected[:, :2]), 1)
    check["mesh_minus_dem_m"] = (check.mesh_elev_m - check.dem_elev_m).round(1)
    print(f"mesh minus DEM: median {check.mesh_minus_dem_m.median():.0f} m, "
          f"largest {check.mesh_minus_dem_m.abs().max():.0f} m")
check

## 8. Write the files

* `model.tria`: the mesh, for GoFEM
* `model.vtu`: the same mesh, to look at in ParaView
* `model.materials`: conductivity of each material
* `model.mask`: which cells the inversion may change (the earth, material 2)
* `receivers_projected.csv`: the stations, 0.1 m below the surface
* `receivers_paraview.csv`: the stations as x, y, z for ParaView

In [ ]:
paths = M.write_mesh_files(tria, materials, names, receivers_projected, OUT_DIR, free_ids=[M.EARTH_ID])
if FEMTIC:
    M.output_format({"output_format": "femtic", "ocean_mode": OCEAN})    # stops on a thin sheet
    from mtinv_gui.backends import femtic
    paths.update(femtic.write_mesh(tria, materials, OUT_DIR, air_id=M.AIR_ID, free_ids=[M.EARTH_ID],
                                   sea_ids=(M.OCEAN_ID,)))
    del paths["femtic_folded"]
if SIMPEG:
    M.output_format({"output_format": "simpeg", "ocean_mode": OCEAN})
    from mtinv_gui.backends import simpeg_mesh
    paths.update(simpeg_mesh.write_mesh(
        tria, materials, OUT_DIR, dem=dem,
        ids=dict(air=M.AIR_ID, earth=M.EARTH_ID, sea=M.OCEAN_ID, mantle=M.MANTLE_ID),
        free_ids=[M.EARTH_ID], land=USE_TOPOGRAPHY and dem is not None, bathymetry=(OCEAN == "real"),
        mantle_depth_m=MANTLE_DEPTH_KM * 1e3 if MANTLE_DEPTH_KM else None,
        names=names, receivers=receivers, vertical_ratio=SIMPEG_VERTICAL_RATIO))
    del paths["simpeg_cells"]
for kind, path in paths.items():
    print(f"{kind:20s} {path}  ({os.path.getsize(path) / 1e6:.1f} MB)")

## 9. Look at the finished model

The starting model's resistivity, with the ocean in it: a plan view just below sea level and the
same section as before, now following the topography.

In [ ]:
cells = mesh_plots.mesh_cells(tria, materials)
fig, (ax_plan, ax_sec) = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={"width_ratios": [1, 1.5]})
mesh_plots.plot_plan(cells, 200.0, ax=ax_plan, stations=receivers_projected, vmin=-1, vmax=3,
                     extent_km=((centre_e - half_e) / 1e3 - 150, (centre_e + half_e) / 1e3 + 150,
                                (centre_n - half_n) / 1e3 - 150, (centre_n + half_n) / 1e3 + 150))
mesh_plots.plot_section(cells, north_m=centre_n, ax=ax_sec, stations=receivers_projected, vmin=-1, vmax=3,
                        range_km=((centre_e - half_e) / 1e3 - 150, (centre_e + half_e) / 1e3 + 150),
                        depth_km=(-3, 20))
plt.show()

## Next

* **Data:** select the data in `gateway_mt` and write the GoFEM data file there, with
  `receivers_projected.csv` from here. The station names in the data must match the names in this
  file. Data Plot also writes FEMTIC's `observe.dat`, taking the station positions from
  `receivers_projected.csv`.
* **SimPEG:** `mesh = discretize.TreeMesh.read_UBC("octree_mesh.txt")`; `simpeg_model.npz` holds the
  starting conductivity, the cell types, the active (inverted) cells and the receiver positions in
  the same cell order (x east, y north, z up).
* **Parameter files:** the GoFEM `.prm` files point at the mesh, the receivers, the frequencies
  and the data file, so write them together with the data (or with the setup GUI's *GoFEM Files*
  tab, from a project using this mesh folder).
* **Look in 3-D:** open `model.vtu` and `receivers_paraview.csv` in ParaView.
* **After the inversion:** notebook 04 refines the result, notebooks 05 and 06 plot it. Without
  an inversion yet, they use a synthetic stand-in made from this mesh.

**For coders:** every step above is a function in `mtinv_gui/backends/mesh.py` (`read_mesh_dem`,
`base_grid`, `initial_materials`, `fit_topography`, `add_ocean`, `project_receivers`,
`write_mesh_files`), and `mesh.run` calls them in the same order for the GUI. The DEM and the sea
mask are `mtinv_gui/backends/mesh_dem.py`. The refinements are pyGoFEM's `gofem.mesh_utils`
functions; `help(refine_around_points)` shows their options.